# Dataset

Lista de puntajes obtenidos por diferentes jugadores oficialmente reconocidos por la Federación Internacional de Ajedrez. Los puntajes son mensuales y se cuenta con información de todos los meses del 2025. Se han considerado solo los puntajes del modo de juego "Blitz". Se descargaron 12 archivos XML con los puntajes, que luego fueron integrados en 1 solo archivo.

Fuente: https://ratings.fide.com/download_lists.phtml

In [ ]:
import pandas as pd

df = pd.read_parquet('data/combined.parquet')

# Data profiling

## Estructura del dataset

Se cuenta con 14 variables y 3 149 829 de observaciones. Hay variables de tipo numérico y cadena de texto.

|Nombre|Tipo|Descripción|
|---|---|---|
|fideid|Int|Número de identificación del jugador en la base de datos de la FIDE|
|name|str|Nombre del jugador|
|country|str|Federación nacional a la que pertenece el jugador|
|sex|str|Sexo del jugador (M = masculino, F = femenino)|
|title|str|Título de ajedrez OTB (*over the board*) del jugador (GM, IM, FM, CM, WGM, WIM, WFM, WCM)|
|w_title|str|Título femenino OTB específico (WGM, WIM, WFM, WCM); presente en jugadoras que poseen tanto un título estándar como uno femenino|
|o_title|str|Otros títulos del jugador: árbitro, entrenador u organizador (IA, FA, FT, FST, entre otros)|
|foa_title|str|Título obtenido en la FIDE Online Arena, plataforma de ajedrez en línea de la FIDE (AFM, AIM, AGM, entre otros)|
|rating|Int|Rating Elo Blitz del jugador|
|games|Int|Número de partidas Blitz jugadas en el período correspondiente|
|k|Int|Factor K del sistema Elo: 40 = jugador nuevo (menos de 30 partidas o menor de 18 años), 20 = estándar, 10 = élite (alguna vez superó los 2400)|
|birthday|float|Año de nacimiento del jugador|
|flag|str|Estado de actividad y sexo codificados: i = inactivo, w = mujer activa, wi = mujer inactiva, NaN = hombre activo|
|filename|str|Nombre del archivo fuente que identifica el mes del registro|


## Calidad del dataset

El dataset presenta buena calidad en sus variables principales (`rating`, `games`, `k`, `fideid`, `name`, `country`, `sex`). Los valores ausentes se concentran en columnas de títulos. La columna `flag` también presenta un alto porcentaje de NaN, que se trabajará en el pre-procesamiento ya que tiene un significado especial.

### Completitud

Las columnas de título (`title`, `w_title`, `o_title`, `foa_title`) presentan entre 93% y 99% de valores ausentes. Esto es esperado ya que solo los jugadores que han obtenido un título oficial de la FIDE los registran. La columna `flag` muestra un 34.64% de NaN, pero esto tampoco representa un problema de completitud ya que corresponde a información mal codificada. El único problema real de completitud es `birthday`, con 1.13% de registros (35,751) sin año de nacimiento.

In [2]:
# get count and percentage of missing data for each column
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100
# create a dataframe to display the results
missing_df = pd.DataFrame({'Missing Count': missing_data, 'Missing Percentage': missing_percentage})
# drop columns with 0 missing rows
missing_df = missing_df[missing_df['Missing Count'] > 0]
print(missing_df)


           Missing Count  Missing Percentage
title            2954110           93.786331
w_title          3110641           98.755838
o_title          3114808           98.888130
foa_title        3091700           98.154504
birthday           35751            1.135014
flag             1091180           34.642505


### Unicidad

El dataset es de tipo panel, donde cada fila representa a un jugador en un mes específico. Existen 283,291 jugadores únicos (`fideid`) que aparecen hasta 12 veces cada uno, pero sin repetirse en el mismo mes. La combinación `fideid` + `flag` presenta 2 808 353 coincidencias, lo que refleja que un mismo jugador mantiene el mismo estado de actividad durante varios meses consecutivos.

In [3]:
# count duplicates across all columnas, with exception of filename
duplicate_count = df.drop(columns=['filename']).duplicated().sum()
print(f'Number of duplicate rows (excluding filename): {duplicate_count}')

Number of duplicate rows (excluding filename): 0


In [4]:
# count unique fideid values
unique_fideid_count = df['fideid'].nunique()
print(f'Number of unique fideid values: {unique_fideid_count}')

Number of unique fideid values: 283291


### Validez

Se verificó que los valores de las variables numéricas y categóricas se encuentren dentro de los rangos esperados. El `rating` está entre 1400 y 2889 en el 100% de los registros, en línea con el piso oficial de la FIDE. Los valores de `k` son únicamente 10, 20 y 40 (los tres valores válidos del sistema Elo). Los valores de `sex` son exclusivamente 'M' y 'F', y ningún jugador masculino posee un título femenino (WGM, WIM, WFM, WCM). Resalta que en `birthday` 34 764 registros corresponden a años de nacimiento posteriores a 2015 (hasta 2021), lo que implicaría jugadores de entre 4 y 9 años de edad. Aunque inusual, podría tratarse de datos válidos.

In [5]:
# Rating bounds
print(f"Rating range: {df['rating'].min()} - {df['rating'].max()}")
print(f"Ratings outside [1400, 3000]: {((df['rating'] < 1400) | (df['rating'] > 3000)).sum()}")

# K-factor valid values
print(f"\nK-factor unique values: {sorted(df['k'].unique())}")

# Sex values and cross-check with women's titles
print(f"\nSex unique values: {df['sex'].unique()}")
males_with_women_title = df[(df['sex'] == 'M') & (df['title'].isin(['WGM', 'WIM', 'WFM', 'WCM']))]
print(f"Males with women's title: {len(males_with_women_title)}")

# Invalid birth years
print("\nInvalid birth years (>2015):")
print(df[df['birthday'] > 2015]['birthday'].value_counts().sort_index().to_frame())

Rating range: 1400 - 2889
Ratings outside [1400, 3000]: 0

K-factor unique values: [np.int64(10), np.int64(20), np.int64(40)]

Sex unique values: <StringArray>
['M', 'F']
Length: 2, dtype: str
Males with women's title: 0

Invalid birth years (>2015):
          count
birthday       
2016.0    22373
2017.0     9485
2018.0     2484
2019.0      383
2020.0       27
2021.0       12


### Consistencia

Se analizó la consistencia longitudinal de los atributos de cada jugador a lo largo de los 12 meses. Se encontraron 905 jugadores con diferencias en su nombre entre meses, probablemente debido a registros manuales. Asimismo, 338 jugadores cambian de `country` entre meses y 76 jugadores presentan valores distintos en `sex` entre meses. No se tiene certeza de si estos cambios son permitidos por la FIDE.

In [6]:
# Name inconsistencies across months
name_per_player = df.groupby('fideid')['name'].nunique()
print(f"Players with >1 name across months: {(name_per_player > 1).sum()}")

# Country changes
country_per_player = df.groupby('fideid')['country'].nunique()
print(f"\nPlayers with >1 country across months: {(country_per_player > 1).sum()}")

# Sex inconsistencies
sex_per_player = df.groupby('fideid')['sex'].nunique()
print(f"\nPlayers with >1 sex value across months: {(sex_per_player > 1).sum()}")

Players with >1 name across months: 905

Players with >1 country across months: 338

Players with >1 sex value across months: 76


# Preprocesamiento

Se aplicaron cinco decisiones de limpieza sobre el dataset original:

1. Eliminar columna `Unnamed: 0`. Columna residual generada al exportar el CSV desde pandas. Duplica el índice sin aportar información.
2. Consolidar columnas de título. Se fusionan `title` y `w_title` en una sola columna `title`, tomando `title` como valor primario y `w_title` como alternativa cuando `title` es nulo. `o_title` (árbitros, entrenadores) y `foa_title` (arena online) corresponden a categorías distintas a los títulos de juego y no son relevantes para el análisis de ratings, por lo que se eliminan.
3. Extraer mes desde `filename`. La columna `filename` codifica el mes en formato `blitz_jan25frl_xml.xml`. Se extrae como entero (1–12) en una nueva columna `month` y se elimina `filename`. Sin esta variable el análisis temporal no es viable.
4. Recodificar `flag` como columna `active`. `flag` codifica simultáneamente el estado de actividad y el sexo del jugador (`i`=inactivo masculino, `w`=femenino activa, `wi`=femenino inactiva, NaN=masculino activo). Se extrae únicamente el componente de actividad como booleano (`active`=True si el jugador estuvo activo en ese período). La información de sexo ya está contenida en la columna `sex`, por lo que no se pierde información al eliminar `flag`.
5. Se mantienen las filas con valores perdidos en `birthday`, pero se renombra la columna a `birth_year`.

In [7]:
print("Estado inicial:")
print(f"  Shape: {df.shape}")
print(f"  Columnas: {list(df.columns)}")
print()
print(df.head(3).to_string())

Estado inicial:
  Shape: (3149830, 15)
  Columnas: ['Unnamed: 0', 'fideid', 'name', 'country', 'sex', 'title', 'w_title', 'o_title', 'foa_title', 'rating', 'games', 'k', 'birthday', 'flag', 'filename']

   Unnamed: 0    fideid                      name country sex title w_title o_title foa_title  rating  games   k  birthday flag                filename
0           0  10245154     A B M Jobair, Hossain     BAN   M   NaN     NaN     NaN       NaN    1748      0  20    1998.0  NaN  blitz_apr25frl_xml.xml
1           1  10207538          A E M, Doshtagir     BAN   M   NaN     NaN     NaN       NaN    1916      0  20    1974.0    i  blitz_apr25frl_xml.xml
2           2  10680810  A hamed Ashraf, Abdallah     EGY   M   NaN     NaN     NaN       NaN    1845      0  20    2001.0    i  blitz_apr25frl_xml.xml


In [8]:
month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
             'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}

# Paso 1: drop Unnamed: 0
df = df.drop(columns=['Unnamed: 0'])

# Paso 2: coalesce title columns, drop w_title / o_title / foa_title
df['title'] = df['title'].fillna(df['w_title'])
df = df.drop(columns=['w_title', 'o_title', 'foa_title'])

# Paso 3: extract month from filename, drop filename
df['month'] = df['filename'].str.extract(r'blitz_([a-z]+)25')[0].map(month_map)
df = df.drop(columns=['filename'])

# Paso 4: flag → active boolean, drop flag
# active=True: flag is NaN (active male) or 'w' (active female)
# active=False: flag is 'i' (inactive male) or 'wi' (inactive female)
df['active'] = ~df['flag'].isin(['i', 'wi'])
df = df.drop(columns=['flag'])

# Paso 5: rename birthday to birth_year
df = df.rename(columns={'birthday': 'birth_year'})

In [9]:
print("Estado final:")
print(f"  Shape: {df.shape}")
print(f"  Columnas: {list(df.columns)}")
print()
print(df.head(3).to_string())

Estado final:
  Shape: (3149830, 11)
  Columnas: ['fideid', 'name', 'country', 'sex', 'title', 'rating', 'games', 'k', 'birth_year', 'month', 'active']

     fideid                      name country sex title  rating  games   k  birth_year  month  active
0  10245154     A B M Jobair, Hossain     BAN   M   NaN    1748      0  20      1998.0      4    True
1  10207538          A E M, Doshtagir     BAN   M   NaN    1916      0  20      1974.0      4   False
2  10680810  A hamed Ashraf, Abdallah     EGY   M   NaN    1845      0  20      2001.0      4   False


# Transformaciones

Se aplicaron dos transformaciones sobre el dataset preprocesado, ambas orientadas a habilitar el análisis exploratorio por grupos.

1. `age`: A partir de `birth_year` se calcula `age = 2025 - birth_year`. Los registros sin `birth_year` producen `age` nulo, lo que es aceptable: estos jugadores quedan fuera de los análisis que requieran esta variable.
2. `age_group`: Sobre `age` se aplica un binning en siete tramos: ≤14, 15-18, 19-25, 26-35, 36-50, 51-65 y 65+, para ser usados a modo de comparación.
3. `rating_category`: El `rating` Elo es continuo (1400–2800+). Se definen cinco niveles basados en umbrales del ajedrez competitivo: Principiante (<1500), Intermedio (1500–1800), Avanzado (1800–2100), Experto (2100–2400) y Élite (≥2400), para aplicar comparaciones.

In [10]:
# T1: age (variable derivada)
df['age'] = (2025 - df['birth_year']).astype('Int64')

# T2: age_group (binning)
age_bins   = [0, 14, 18, 25, 35, 50, 65, 200]
age_labels = ['≤14', '15-18', '19-25', '26-35', '36-50', '51-65', '65+']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)

# T3: rating_category (binning)
rating_bins   = [0, 1500, 1800, 2100, 2400, 3000]
rating_labels = ['Principiante', 'Intermedio', 'Avanzado', 'Experto', 'Élite']
df['rating_category'] = pd.cut(df['rating'], bins=rating_bins, labels=rating_labels)

In [11]:
print("age_group distribution:")
print(df['age_group'].value_counts().sort_index().to_frame())
print()
print("rating_category distribution:")
print(df['rating_category'].value_counts().sort_index().to_frame())
print()
print(f"Columnas finales: {list(df.columns)}")

age_group distribution:
            count
age_group        
≤14        387721
15-18      482211
19-25      620328
26-35      464416
36-50      519596
51-65      394350
65+        245457

rating_category distribution:
                   count
rating_category         
Principiante      426039
Intermedio       1578990
Avanzado          938214
Experto           185427
Élite              21160

Columnas finales: ['fideid', 'name', 'country', 'sex', 'title', 'rating', 'games', 'k', 'birth_year', 'month', 'active', 'age', 'age_group', 'rating_category']


In [13]:
# compute records by country
country_counts = df['country'].value_counts()

# Análisis Exploratorio de Datos

Se plantean cuatro hipótesis sobre el comportamiento esperado del dataset, evaluadas con tablas de resumen.

**H1**: La mayoría de los jugadores registrados están activos.

In [82]:
# H1: La mayoría de los jugadores está activo
print(df['active'].value_counts()
      .to_frame('registros')
      .assign(porcentaje=lambda x: (x['registros'] / x['registros'].sum() * 100).round(1)))

        registros  porcentaje
active                       
False     1916249        60.8
True      1233581        39.2


**Rechazada**. Solo el 39.2% de los registros corresponde a jugadores activos; el 60.8% está marcado como inactivo por la FIDE.

**H2**: El rating promedio y mediano no varía a lo largo del año.

In [83]:
# H2: El rating no varía a lo largo del año
print(df.groupby('month')['rating'].agg(media='mean', mediana='median').round(1))

        media  mediana
month                 
1      1748.3   1725.0
2      1747.2   1723.0
3      1746.2   1722.0
4      1745.1   1720.0
5      1743.6   1718.0
6      1742.5   1717.0
7      1741.1   1715.0
8      1738.5   1712.0
9      1737.2   1710.0
10     1735.2   1708.0
11     1734.2   1707.0
12     1733.0   1705.0


**Rechazada**. La media cae de 1748 (enero) a 1733 (diciembre) y la mediana de 1725 a 1705 — un descenso sistemático de ~15 puntos. El patrón refleja la incorporación mensual de nuevos jugadores con ratings más bajos, no el deterioro de los ya registrados.

**H3**: La proporción de jugadores inactivos es mayor en los niveles de rating más bajos.

In [84]:
# H3: Mayor proporción de inactivos en ratings bajos
print(df.groupby('rating_category', observed=True)['active']
      .agg(total='count',
           pct_activo=lambda x: round(x.mean() * 100, 1),
           pct_inactivo=lambda x: round((~x).mean() * 100, 1)))

                   total  pct_activo  pct_inactivo
rating_category                                   
Principiante      426039        45.3          54.7
Intermedio       1578990        39.2          60.8
Avanzado          938214        35.3          64.7
Experto           185427        42.7          57.3
Élite              21160        55.0          45.0


**Rechazada**. La mayor tasa de inactividad se encuentra en el nivel Avanzado (64.7%), no en Principiante (54.7%). Los jugadores de Élite tienen la menor tasa (45.0%), y los de Principiante incluyen muchos nuevos jugadores activamente compitiendo para establecer su rating.

**H4**: Los jugadores de mayor edad tienen ratings más altos.

In [85]:
# H4: Jugadores mayores tienen ratings más altos
print(df.groupby('age_group', observed=True)['rating']
      .agg(mediana='median', media='mean', n='count')
      .round(1))

           mediana   media       n
age_group                         
≤14         1531.0  1566.2  387721
15-18       1584.0  1626.1  482211
19-25       1678.0  1707.2  620328
26-35       1790.0  1808.7  464416
36-50       1818.0  1832.1  519596
51-65       1841.0  1847.1  394350
65+         1819.0  1820.8  245457


**Parcialmente confirmada**. El rating mediano crece con la edad hasta el tramo 51-65 (mediana 1841), pero desciende en el grupo 65+ (1819). Los jugadores más jóvenes (≤14) tienen la mediana más baja (1531).